# Processamento do Dataset visual-memory/PersonaChat

Este notebook processa todos os splits de `visual-memory/PersonaChat` para:
1. Normalizar as personas originais e revisadas, armazenadas como listas JSON
2. Gerar IDs SHA-256 determinísticos a partir das personas originais normalizadas
3. Criar um **dataset enriquecido** com `your_persona_id` e `partner_persona_id`
4. Criar um **dataset de mapeamento** com as personas originais e revisadas
5. Validar a integridade dos datasets antes da publicação

In [11]:
import hashlib
import json
import re
from typing import Any

from datasets import Dataset, DatasetDict, load_dataset

PERSONA_FIELDS = (
    {
        "role": "your",
        "original": "your_persona_original",
        "revised": "your_persona_revised",
        "enhanced_original": "your_enhanced_persona_original",
        "enhanced_revised": "your_enhanced_persona_revised",
        "id": "your_persona_id",
    },
    {
        "role": "partner",
        "original": "partner_persona_original",
        "revised": "partner_persona_revised",
        "enhanced_original": "partner_enhanced_persona_original",
        "enhanced_revised": "partner_enhanced_persona_revised",
        "id": "partner_persona_id",
    },
)
REQUIRED_COLUMNS = {
    field[column]
    for field in PERSONA_FIELDS
    for column in ("original", "revised", "enhanced_original", "enhanced_revised")
}
SHA256_PATTERN = re.compile(r"^[0-9a-f]{64}$")


def normalize_persona(value: Any, *, column: str, row_context: str) -> str:
    """Converte uma lista JSON de frases em um texto normalizado."""
    if not isinstance(value, str):
        raise ValueError(
            f"Valor invalido em {column} ({row_context}): esperado str JSON, "
            f"recebido {type(value).__name__}."
        )

    try:
        phrases = json.loads(value)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"JSON invalido em {column} ({row_context}): {exc.msg}."
        ) from exc

    if not isinstance(phrases, list):
        raise ValueError(
            f"Valor invalido em {column} ({row_context}): esperado uma lista JSON."
        )

    invalid_items = [
        index for index, phrase in enumerate(phrases) if not isinstance(phrase, str)
    ]
    if invalid_items:
        raise ValueError(
            f"Itens invalidos em {column} ({row_context}): os indices "
            f"{invalid_items} nao contem strings."
        )

    normalized_phrases = [phrase.strip() for phrase in phrases if phrase.strip()]
    if not normalized_phrases:
        raise ValueError(
            f"Persona vazia em {column} ({row_context}): nenhuma frase valida encontrada."
        )

    return " ".join(normalized_phrases)


def normalize_enhanced_persona(
    value: Any, *, column: str, row_context: str
) -> str:
    """Valida e normaliza um texto aprimorado de persona."""
    if not isinstance(value, str) or not value.strip():
        raise ValueError(
            f"Texto aprimorado invalido em {column} ({row_context}): "
            "esperado texto nao vazio."
        )
    return value.strip()


def generate_persona_id(persona_original: str) -> str:
    """Gera um ID SHA-256 determinístico para a persona original normalizada."""
    return hashlib.sha256(persona_original.encode("utf-8")).hexdigest()

## 1. Carregar e validar o dataset (todos os splits)

In [12]:
dataset_dict = load_dataset("visual-memory/PersonaChat")

missing_columns = {
    split_name: sorted(REQUIRED_COLUMNS - set(dataset.column_names))
    for split_name, dataset in dataset_dict.items()
}
missing_columns = {split: columns for split, columns in missing_columns.items() if columns}
if missing_columns:
    raise ValueError(f"Colunas de persona ausentes por split: {missing_columns}")

print(f"Total de linhas (todos os splits): {sum(len(ds) for ds in dataset_dict.values())}")
for split_name, dataset in dataset_dict.items():
    print(f"  {split_name}: {len(dataset)} linhas")

Total de linhas (todos os splits): 10907
  train: 8939 linhas
  validation: 1000 linhas
  test: 968 linhas


## 2. Normalizar e unificar os registros de persona

In [13]:
personas_by_original: dict[str, dict[str, str]] = {}

for split_name, dataset in dataset_dict.items():
    for row_index, row in enumerate(dataset):
        row_context = f"split={split_name!r}, row={row_index}"
        for fields in PERSONA_FIELDS:
            persona_original = normalize_persona(
                row[fields["original"]],
                column=fields["original"],
                row_context=row_context,
            )
            persona_data = {
                "persona_original": persona_original,
                "persona_revised": normalize_persona(
                    row[fields["revised"]],
                    column=fields["revised"],
                    row_context=row_context,
                ),
            }

            previous_data = personas_by_original.get(persona_original)
            if previous_data is not None and previous_data != persona_data:
                conflicting_fields = [
                    key for key, value in persona_data.items()
                    if previous_data[key] != value
                ]
                raise ValueError(
                    f"Associacao conflitante para persona em {row_context}, "
                    f"role={fields['role']!r}: {conflicting_fields}."
                )
            personas_by_original[persona_original] = persona_data

sorted_original_personas = sorted(personas_by_original)
print(f"Personas originais unicas: {len(sorted_original_personas)}")
print(f"Exemplo de persona normalizada: {sorted_original_personas[0]}")

Personas originais unicas: 21061
Exemplo de persona normalizada: 1984 is my favorite book. i am allergic to nuts. i am working on a biology degree. i am in college. i love book.


## 3. Gerar IDs determinísticos (SHA-256)

In [14]:
original_to_id = {
    persona_original: generate_persona_id(persona_original)
    for persona_original in sorted_original_personas
}

if len(set(original_to_id.values())) != len(original_to_id):
    raise ValueError("Colisao de IDs SHA-256 detectada entre personas diferentes.")

print(f"IDs gerados: {len(original_to_id)}")
print(f"Exemplo de ID: {next(iter(original_to_id.values()))}")

IDs gerados: 21061
Exemplo de ID: 4f657d415a70b44862d244da8a358d45f0cab4ec89b57b6f6b1be3ee0d13cf7a


## 4. Criar o dataset enriquecido

In [15]:
enriched_dataset_dict = DatasetDict()

for split_name, dataset in dataset_dict.items():
    enriched_rows = []
    for row_index, row in enumerate(dataset):
        row_context = f"split={split_name!r}, row={row_index}"
        enriched_row = dict(row)
        for fields in PERSONA_FIELDS:
            persona_original = normalize_persona(
                row[fields["original"]],
                column=fields["original"],
                row_context=row_context,
            )
            enriched_row[fields["id"]] = original_to_id[persona_original]
        enriched_rows.append(enriched_row)

    enriched_dataset_dict[split_name] = Dataset.from_list(enriched_rows)

print("Dataset enriquecido (estrutura DatasetDict):")
for split_name, dataset in enriched_dataset_dict.items():
    print(f"  {split_name}: {len(dataset)} linhas")
first_split = next(iter(enriched_dataset_dict))
print(f"Colunas: {enriched_dataset_dict[first_split].column_names}")

Dataset enriquecido (estrutura DatasetDict):
  train: 8939 linhas
  validation: 1000 linhas
  test: 968 linhas
Colunas: ['dialog_id', 'your_persona_original', 'partner_persona_original', 'your_persona_revised', 'partner_persona_revised', 'dialog', 'your_enhanced_persona_original', 'partner_enhanced_persona_original', 'your_enhanced_persona_revised', 'partner_enhanced_persona_revised', 'your_persona_id', 'partner_persona_id']


## 5. Criar o dataset de mapeamento

In [16]:
mapping_data = []
for persona_original in sorted_original_personas:
    persona_data = personas_by_original[persona_original]
    mapping_data.append(
        {
            "persona-id": original_to_id[persona_original],
            "persona_original": persona_data["persona_original"],
            "persona_revised": persona_data["persona_revised"],
        }
    )

mapping_dataset = Dataset.from_list(mapping_data)

print(f"Dataset de mapeamento: {len(mapping_dataset)} personas unicas")
print(f"Colunas: {mapping_dataset.column_names}")

Dataset de mapeamento: 21061 personas unicas
Colunas: ['persona-id', 'persona_original', 'persona_revised']


## 6. Validar a integridade dos datasets

In [17]:
assert set(enriched_dataset_dict) == set(dataset_dict), "Os splits foram alterados."
for split_name, original_dataset in dataset_dict.items():
    enriched_dataset = enriched_dataset_dict[split_name]
    assert len(enriched_dataset) == len(original_dataset), (
        f"Quantidade de linhas alterada no split {split_name!r}."
    )
    assert enriched_dataset.column_names == [
        *original_dataset.column_names,
        "your_persona_id",
        "partner_persona_id",
    ], f"Colunas inesperadas no split {split_name!r}."

expected_mapping_columns = [
    "persona-id",
    "persona_original",
    "persona_revised",
]
assert mapping_dataset.column_names == expected_mapping_columns

mapping_by_id = {}
for mapping_row in mapping_dataset:
    persona_id = mapping_row["persona-id"]
    assert SHA256_PATTERN.fullmatch(persona_id), f"ID invalido no mapping: {persona_id!r}"
    assert persona_id not in mapping_by_id, f"ID duplicado no mapping: {persona_id!r}"
    assert persona_id == generate_persona_id(mapping_row["persona_original"]), (
        f"ID inconsistente com persona_original: {persona_id!r}"
    )
    mapping_by_id[persona_id] = mapping_row

for split_name, dataset in enriched_dataset_dict.items():
    for row_index, row in enumerate(dataset):
        row_context = f"split={split_name!r}, row={row_index}"
        for fields in PERSONA_FIELDS:
            persona_id = row[fields["id"]]
            assert isinstance(persona_id, str) and SHA256_PATTERN.fullmatch(persona_id), (
                f"ID invalido em {row_context}, column={fields['id']!r}: {persona_id!r}"
            )
            assert persona_id in mapping_by_id, (
                f"ID sem mapping em {row_context}, column={fields['id']!r}: {persona_id!r}"
            )

assert len(mapping_by_id) == len(personas_by_original)
print("Todas as validacoes foram concluidas com sucesso.")

sample = enriched_dataset_dict[first_split][0]
print(f"\nExemplo de linha enriquecida (split {first_split!r}):")
print(f"  your_persona_id: {sample['your_persona_id']}")
print(f"  partner_persona_id: {sample['partner_persona_id']}")

Todas as validacoes foram concluidas com sucesso.

Exemplo de linha enriquecida (split 'train'):
  your_persona_id: d8fa91ccb29532992383fdbf0cc98af8d944132f68aa2f7dd3b1fd77c5f77470
  partner_persona_id: fc7f99c395b916c3c90efa1bd1741534900f16c140ef03e1a6a783aea732bba5


## 7. Publicar no Hugging Face Hub

> Execute as duas células abaixo manualmente, somente depois de revisar as validações.

In [18]:
from huggingface_hub import notebook_login

notebook_login()

In [20]:
enriched_dataset_dict.push_to_hub("visual-memory/PersonaChat-With-Ids")
mapping_dataset.push_to_hub("visual-memory/PersonaChat-Mapping")

print("Dataset enriquecido publicado!")
print("Dataset de mapeamento publicado!")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Dataset enriquecido publicado!
Dataset de mapeamento publicado!
